# MyCON — تنها نوت‌بوک نهایی با سوییچ امن CPU → GPU

**روش اصلی فعلی:** فیلم واقعی Recorder → COLMAP → نسخه رسمی gsplat 1.5.3 → مدل و رندر و معیارهای واقعی. NVIDIA LongSplat بدون حذف checkpointها به‌عنوان شاخه پژوهشی مقایسه‌ای محفوظ است.

۱. Colab Runtime را **CPU** انتخاب کن و **Run all** بزن؛ ZIP بررسی و COLMAP ساخته می‌شود، سپس با پیام INTENTIONAL_CPU_STOP عمداً متوقف می‌شود.
۲. Runtime را دستی به **T4 GPU** تغییر بده؛ با این کار `/content` پاک می‌شود اما تمام رسیدها و داده‌های ساخته‌شده روی **Google Drive** محفوظ‌اند.
۳. دوباره **Run all از اول** بزن. نوت‌بوک CPU را خودکار Skip می‌کند، فریم‌های COLMAP را از Drive بازیابی می‌کند و یک پیش‌اجرای واقعی CUDA و فقط **۲۰ مرحله کوچک روی همان فیلم** انجام می‌دهد.
۴. آموزش ۷۰۰۰ مرحله‌ای **پیش‌فرض خاموش** است. برای اجازه صریح، `TRAIN_CONFIRMATION = "RUN_7000_STEPS_T4"` را در اولین سلول تغییر بده؛ بعد از preflight موفق، همان Run all روی T4 آموزش را اجرا می‌کند.
۵. اگر GPU وسط آموزش قطع شد، همان ATTEMPT را حفظ کن: snapshot کامل Gaussian+optimizer+scheduler+densification پس از مرحله ۲۰۰ و سپس هر ۵۰۰ مرحله روی Drive ذخیره می‌شود و Run all بعدی می‌تواند ادامه دهد؛ تا مرحله ۲۰۰ هنوز checkpoint بازیابی کامل نداریم.

**توجه:** این طراحی آموزش را در یک Runtime ادامه می‌دهد، اما پس از قطع‌شدن Colab، محیط Python/CUDA باید روی T4 جدید دوباره نصب/تأیید شود. PLY به‌تنهایی برای اعلام پیشرفت BIM کافی نیست.

**نسخه ممیزی‌شده:** بازیابی قطع FFmpeg/COLMAP، هش کامل فریم‌ها، قفل SHA کد در CPU→GPU، مدل و ارزیابی canonical، محدودکننده رشد Gaussian/VRAM، و بررسی رسمی SHA wheel gsplat. فایل‌های ورودی هرگز بازنویسی نمی‌شوند.

**برای BIM:** کاندید sparse تنها پس از تأمین IFC واقعی و نقاط کنترل مستقل می‌تواند به `scripts/mycon_colmap_bim_adapter.py` وارد شود؛ هیچ مرحله پیشرفت Stage9/Stage11 تا تأیید هندسی فعال نمی‌شود.

**قفل نسخه:** این نوت‌بوک کدهای اجرایی را روی commit `b7649b36b283` نگه می‌دارد تا تعویض CPU→T4 و به‌روزرسانی احتمالی شاخه GitHub هیچ تغییری در وسط آزمایش ایجاد نکند.

**گیت جدید G1:** تشخیص کیفیت فریم‌های استخراج‌شده و ارزیابی ۸ بخش زمانی و بزرگ‌ترین فاصلهٔ بدون ثبت دوربین؛ نتیجه نامعتبر جلوی ورود به GPU را می‌گیرد. فریم‌های اصلی حذف نمی‌شوند. این کار هنوز جای انتخاب تطبیقی فریم از ویدئوی پرنرخ، سنجش پارالاکس واقعی، یا ارزیابی هندسهٔ BIM نیست.

**دو ورودی مستقل در CPU:** پیش از COLMAP، تصویرهای ۸fps در عرض ۵۱۲ برای LongSplat، جداگانه و با هش تمام فریم‌ها آماده می‌شوند. بعد مسیر ۲fps/۶۴۰ COLMAP اجرا می‌شود. **هیچ آموزش LongSplat، مرحله CUDA یا بازیابی checkpoint قدیمی به‌طور خودکار اجرا نمی‌شود.**

**ممیزی تطبیقی نوت‌بوک‌های واقعی (۶ نوامبر):** سازگاری PyTorch3D 0.7.8 با torch 2.4.1 / CUDA 12.1 در ابزار جداگانهٔ NVIDIA اصلاح شده و آدرس محیط جدید از نسخه قدیمی جداست. این نوت‌بوک همچنان روی COLMAP+gsplat baseline با ورودی LongSplat صرفاً برای dataset عمل می‌کند، نه آموزش NVIDIA؛ هیچ ادعای تضمین عدم خطای Runtime یا آموزش کامل از تست آفلاین ارائه نمی‌شود.

**توجه کاربران دارای CPU موفق نسخه v6:** برای ادامه بدون تکرار CPU باید دفترچهٔ جداگانهٔ `MyCON_GPU_RECOVERY_v7_20261106.ipynb` اجرا شود. این دفترچهٔ عمومی نسخه v7 برای آزمایش تازه است. خطای مشاهده‌شدهٔ قبلی از مقایسهٔ نادرست رشتهٔ نسخه gsplat ناشی شده بود که اصلاح شده؛ تست CUDA واقعی بعد از اصلاح هنوز نیاز به اجرا دارد.

**اصلاح خطای دوم v7 (MPLBACKEND):** متغیر محیطی `module://matplotlib_inline.backend_inline` در محیط Python مجزای CUDA موجب خطای `torchmetrics` می‌شد. در نسخه v8 فقط subprocessهای gsplat و LongSplat به `Agg` هدایت می‌شوند. **این نوت‌بوک یک آزمایش جدید است؛ برای ادامه همان CPU موفق قدیمی از نوت‌بوک بازیابی MyCON_GPU_RECOVERY_v8 استفاده کن.**

**افزودهٔ ممیزی حافظه:** در استفادهٔ مجدد از کش GPU، شرط فضای دیسک بر مبنای فایل‌های واقعاً کپی‌نشده و ۵۱۲MiB حاشیهٔ اطمینان محاسبه می‌شود؛ نصب جدید CUDA همچنان نیاز به فضای مستقل دارد. برای CPU موفق v6 قبلی از دفترچهٔ بازیابی v9 استفاده شود، نه این نوت‌بوک آزمایش جدید.


## تنظیمات، دریافت MyCON و اتصال Drive — در هر دو Runtime خودکار اجرا می‌شود


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, shutil, sys, json

CAPTURE_ZIP = ""  # Only if original Recorder ZIP is not uniquely auto-discovered
ATTEMPT = "mycon_colab_backend_checked_v8_20261106"
TRAIN_CONFIRMATION = ""  # Explicit training: RUN_7000_STEPS_T4
BRANCH = "feature/mycon-single-final-acceptance-20261105"
# Exact validated source version for BOTH CPU and GPU; do not follow moving HEAD
PINNED_SOURCE_SHA = "3627d6a9a51fa2098d1dc490e9364be7c2d2fa4c"
URL = "https://github.com/sayyedalimrj/MyCon.git"

drive.mount("/content/drive")
repo=Path("/content/MyCON_FINAL_ACCEPTANCE")
if repo.exists():
    assert (repo/".git").exists(), "Unexpected existing directory; preserving."
    origin=subprocess.check_output(["git","-C",str(repo),"remote","get-url","origin"],text=True).strip()
    assert origin in (URL,"git@github.com:sayyedalimrj/MyCon.git"),"Wrong repo origin."
    dirty=subprocess.check_output(["git","-C",str(repo),"status","--porcelain"],text=True).strip()
    assert not dirty,"Local edits exist; refusing overwrite."
    subprocess.run(["git","-C",str(repo),"fetch","--depth","1",
                    "origin",PINNED_SOURCE_SHA],check=True)
    subprocess.run(["git","-C",str(repo),"checkout","--detach",PINNED_SOURCE_SHA],check=True)
else:
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,URL,str(repo)],check=True)
    subprocess.run(["git","-C",str(repo),"fetch","--depth","1",
                    "origin",PINNED_SOURCE_SHA],check=True)
    subprocess.run(["git","-C",str(repo),"checkout","--detach",PINNED_SOURCE_SHA],check=True)
script=repo/"repository/scripts/mycon_colmap_gsplat.py"
assert script.is_file(),"Incomplete MyCON source."
sha=subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()
assert sha==PINNED_SOURCE_SHA, "Source lock mismatch. Never mix CPU/GPU releases."
print("MYCON_PINNED_SOURCE_COMMIT:",sha)

def stage(mode,approve=False):
    cmd=[sys.executable,str(script),"--mode",mode,"--attempt",ATTEMPT]
    if CAPTURE_ZIP.strip():cmd+=["--capture",CAPTURE_ZIP.strip()]
    if approve:cmd+=["--approve-training"]
    print("MYCON_STAGE:",mode, "APPROVE_FULL_GPU_TRAINING:",approve,flush=True)
    completed=subprocess.run(cmd,check=False)
    if completed.returncode:
        print("MYCON_STAGE_FAILED:",mode,"exit_code",completed.returncode,flush=True)
        print("Existing Recorder and checkpoints have NOT been deleted.",flush=True)
        print("Inspect the Drive experiment logs and last failure before retrying.",flush=True)
        from collections import deque
        logfiles=sorted((Path("/content/drive/MyDrive/MyCon_R4")/"research_colmap_gsplat").glob("*/"+ATTEMPT+"/logs/pipeline.log"))
        if len(logfiles)==1:
            with logfiles[0].open(encoding="utf-8",errors="replace") as logfh:
                tail="".join(deque(logfh,maxlen=120))
            at=tail.rfind("Traceback (most recent call last)")
            print("ACTUAL_FAILURE_IN_PIPELINE_LOG:\n"+(tail[at:] if at>=0 else tail[-12000:]),flush=True)
            print("FULL_LOG_PATH:",logfiles[0],flush=True)
        if mode!="status":
            subprocess.run([sys.executable,str(script),"--mode","status",
                            "--attempt",ATTEMPT],check=False)
        raise RuntimeError("MYCON_STAGE_ABORTED_SAFELY: "+mode)

def gpu_info():
    if not shutil.which("nvidia-smi"):return None
    p=subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap",
                      "--format=csv,noheader"],capture_output=True,text=True)
    return p.stdout.strip() if p.returncode==0 and p.stdout.strip() else None


## مرحله CPU — ساخت فقط یک‌بار روی Google Drive و سپس توقف عمدی Run all


In [ ]:
gpu=gpu_info()
if gpu is None:
    print("CPU_ONLY: verifying immutable Recorder ZIP and building COLMAP geometry.",flush=True)
    stage("verify")
    if any(shutil.which(x) is None for x in ("ffmpeg","ffprobe","colmap")):
        subprocess.run(["apt-get","update","-qq"],check=True)
        subprocess.run(["apt-get","install","-y","ffmpeg","colmap"],check=True)
    stage("longsplat-dataset")  # Safe 8fps/512 CPU input; NO NVIDIA training
    stage("geometry")
    print("CPU_GEOMETRY_COMPLETE_ON_GOOGLE_DRIVE. SWITCH RUNTIME TO NVIDIA T4.",flush=True)
    print("INTENTIONAL_CPU_STOP_NO_EXCEPTION: CPU work saved on Drive; switch Runtime to T4 and Run all again.",flush=True)
else:
    if "T4" not in gpu or "7.5" not in gpu:
        raise RuntimeError("WRONG_GPU_SELECT_T4: "+gpu)
    print("GPU_T4_DETECTED: CPU verify and COLMAP stages SKIPPED automatically.",flush=True)


## مرحله T4 — پیش‌اجرای واقعی ۲۰ مرحله‌ای، سپس آموزش فقط با تأیید


In [ ]:
if gpu_info() is None:
    print("CPU_RUNTIME_FINISHED: GPU preflight skipped without an intentional exception. Switch to T4.",flush=True)
else:
    stage("preflight")
    print("Preflight includes a REAL 20-step video/optimizer test, not full 7000-step training.")
    print("REQUIRED: PREFLIGHT_REAL_VIDEO_GPU_SMOKE_PASSED_NO_FULL_TRAINING")
    if TRAIN_CONFIRMATION == "RUN_7000_STEPS_T4":
        print("Explicit full-training approval present.",flush=True)
        stage("all",approve=True)
    else:
        print("FULL GPU TRAINING BLOCKED. Explicit confirmation required.")


## وضعیت صرفاً خواندنی — هیچ آموزش تازه‌ای آغاز نمی‌کند


In [ ]:
if gpu_info() is not None:
    stage("status")
else:
    print("CPU_SUCCESSFUL: status is safely saved on Drive; GPU not started.")


**خروجی‌های اصلی:** `research_colmap_gsplat/<SHA16>/<ATTEMPT>/` شامل آرشیو رسید، فریم‌ها، `sparse/0`، ابرنقاط هندسی غیرمتریک، snapshotهای بازیابی، Gaussian PLY، ویدئو و شاخص‌های PSNR/SSIM/LPIPS. جهت مقایسه واقعی BIM باید در Stage8 نقاط کنترل مستقل و IFC واقعی داشته باشیم.

نسخه کتابخانه‌ها و نسخه‌های upstream در [قفل سازگاری](docs/MYCON_RUNTIME_COMPATIBILITY_LOCK_20261105.json) آمده است؛ هر دو محیط Python میزبان Colab و Python آموزش جدا هستند.

**خروجی نهاییِ مدل:** `gsplat_result/canonical/` (checkpoint، Gaussian PLY، رندر، معیارها و SHAهای مرتبط). `colmap_sparse_unscaled.ply` هنوز در واحدهای غیرمتریک COLMAP است.

**گزارش‌های QA افزوده‌شده:** `derived_frame_quality_qa.json` و `temporal_coverage_qa.json`. فقط پروندهٔ آزمایش جدید `mycon_colab_backend_checked_v8_20261106` برای این نسخه استفاده شود؛ آزمایش‌های پیشین و checkpointهای LongSplat بازنویسی نمی‌شوند.

**LongSplat input:** `longsplat_video8fps_512/images/` و `dataset_receipt.json` فقط دادهٔ خام مشتق‌شده و بدون ارزیابی مدل هستند، نه خروجی نهایی NVIDIA. اجرای اصلی LongSplat و تبدیل 3DGS هنوز به ارزیابی جداگانه نیاز دارد.
